In [ ]:
import os
import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
import math

import random
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import matplotlib.dates as mdates
from scipy import stats
import itertools
import matplotlib.pyplot as plt

In [ ]:
parent_dir = "../../Data"

In [ ]:
machine_data_raw= pd.read_csv(os.path.join(parent_dir, "Intermediate", "extracted_raw.csv"), index_col=False)
mortality_label=pd.read_csv(os.path.join(parent_dir, "Intermediate", "mortality_label.csv"), index_col=False) 
merged_meta_data=pd.read_csv(os.path.join(parent_dir, "Intermediate", "Events_date_time.csv"), index_col=False) 


In [ ]:
machine_data_raw.columns

In [ ]:
mortality_label.columns

In [ ]:
merged_meta_data.columns

## Filter Machine Data based on window based anomaly detection

In [ ]:
# Algorithm: Temporal Anomaly Filtering for CRRT Machine Data

# Goal:
#   Remove clinically invalid CRRT machine time segments by aligning
#   machine-recorded timestamps with authoritative clinical dates per patient.
#
# Steps:
# 1. Restrict machine data to PIDs present in metadata.
# 2. Standardize all relevant timestamp columns to datetime.
# 3. For each PID, identify anomalous intervals using:
#    - Start rule: If CRRTStartTime_Machine occurs before the reference start
#      (CrrtStartDate → IcuAdmitDate → HospitalAdmitDate), remove early data.
#    - End rule: If CRRTEndTime_Machine exceeds the reference end
#      (CrrtIndependence → IcuLastDate → HospitalDischargeDate) by >12 hours,
#      remove late data (within tolerance is retained).
# 4. Merge overlapping anomaly intervals per PID.
# 5. Exclude machine data rows whose timestamps fall within anomaly intervals.
#
# Output:
#   A cleaned CRRT machine time-series dataset aligned with clinical timelines.


pids_metadata_unique= merged_meta_data['PID'].unique()
machine_data_filtered = machine_data_raw[machine_data_raw['PID'].isin(pids_metadata_unique)]

expected_date_columns = [
    'HospitalAdmitDate',
    'IcuAdmitDate',
    'CrrtStartDate',
    'CRRTStartTime_Machine',
    'CRRTEndTime_Machine',
    'CrrtIndependence',
    'IcuLastDate',
    'HospitalDischargeDate']

# --- Step 1: Standardize datetime columns ---
for c in expected_date_columns:
    if c in merged_meta_data.columns:
        merged_meta_data[c] = pd.to_datetime(merged_meta_data[c], errors="coerce")

machine_data_filtered = machine_data_filtered.copy()
machine_data_filtered["Time"] = pd.to_datetime(machine_data_filtered["Time"], errors="coerce")

TOL = pd.Timedelta(hours=12)


# --- Step 2: Build anomaly intervals for each PID ---
def anomaly_intervals_from_row_v2(row):
    """
    From one PID row in merged_meta_data, produce a list of anomalous (start, end) intervals
    according to the NEW rules:
    
    1. If CRRTStartTime_Machine < reference_start (CrrtStartDate > IcuAdmitDate > HospitalAdmitDate),
       then remove data before reference_start.
    2. If CRRTEndTime_Machine > reference_end:
       - If CRRTEndTime_Machine > reference_end + 12h -> remove data after reference_end
       - Else (within tolerance) -> keep until CRRTEndTime_Machine.
    """
    intervals = []
    
    # --- Rule 1: Start time anomaly ---
    ref_start = row["CrrtStartDate"]
    if pd.isna(ref_start):
        ref_start = row["IcuAdmitDate"]
    if pd.isna(ref_start):
        ref_start = row["HospitalAdmitDate"]

    crrt_start_machine = row.get("CRRTStartTime_Machine", pd.NaT)
    if pd.notnull(ref_start) and pd.notnull(crrt_start_machine):
        if crrt_start_machine < ref_start:
            # anomalous range: (machine_start, ref_start)
            intervals.append((crrt_start_machine, ref_start))

    # --- Rule 2: End time anomaly ---
    ref_end = row["CrrtIndependence"]
    if pd.isna(ref_end):
        ref_end = row["IcuLastDate"]
    if pd.isna(ref_end):
        ref_end = row["HospitalDischargeDate"]

    crrt_end_machine = row.get("CRRTEndTime_Machine", pd.NaT)
    if pd.notnull(ref_end) and pd.notnull(crrt_end_machine):
        if crrt_end_machine > ref_end:
            if crrt_end_machine > ref_end + TOL:
                # anomaly: cut everything after ref_end
                intervals.append((ref_end, crrt_end_machine))
            else:
                # within tolerance → extend normal range up to machine end (no anomaly here)
                pass

    return intervals


# --- Step 3: Build all anomaly intervals per PID ---
all_intervals = []
for _, r in merged_meta_data.iterrows():
    pid = r["PID"]
    for start, end in anomaly_intervals_from_row_v2(r):
        all_intervals.append({"PID": pid, "start": start, "end": end})

intervals_df = pd.DataFrame(all_intervals, columns=["PID", "start", "end"])


# --- Step 4: Merge overlapping intervals ---
if intervals_df.empty:
    filtered_machine = machine_data_filtered.copy()
else:
    merged_intervals = []
    for pid, grp in intervals_df.sort_values(["PID", "start", "end"]).groupby("PID"):
        current_start = None
        current_end = None
        for _, row in grp.iterrows():
            s, e = row["start"], row["end"]
            if current_start is None:
                current_start, current_end = s, e
            else:
                if s <= current_end:  # overlapping
                    if e > current_end:
                        current_end = e
                else:
                    merged_intervals.append({"PID": pid, "start": current_start, "end": current_end})
                    current_start, current_end = s, e
        if current_start is not None:
            merged_intervals.append({"PID": pid, "start": current_start, "end": current_end})

    merged_intervals_df = pd.DataFrame(merged_intervals, columns=["PID", "start", "end"])

    # --- Step 5: Exclude rows falling inside anomalous intervals ---
    mask = pd.Series(False, index=machine_data_filtered.index)
    for _, iv in merged_intervals_df.iterrows():
        pid, s, e = iv["PID"], iv["start"], iv["end"]
        mask |= (machine_data_filtered["PID"].eq(pid) &
                 machine_data_filtered["Time"].between(s, e, inclusive="both"))

    filtered_machine = machine_data_filtered.loc[~mask].copy()

## Priming Time removal

In [ ]:
# Compute visit number for each patient
# Condition:
#   For any patient, mark a new visit if the gap between timestamps is more than 24 hours

def compute_visit_number(df):
    df=df.sort_values(by=['PID','Time'])
    df['Visit_no'] = 0
    for pid,group in df.groupby('PID'):
        group['time_diff'] = group['Time'].diff().dt.total_seconds()
        group['Visit_no'] = (group['time_diff'] > 24*3600).cumsum()+1
        df.loc[group.index,'Visit_no'] = group['Visit_no']
    return df

filtered_merged_df = compute_visit_number(filtered_machine)

machine_filtered_v1 = filtered_merged_df[['PID', 'Visit_no', 'Time', 'RUN_TIME', 'POST_INFUSION', 'PRE_INFUSION', 'DIALYSATE',
       'EFFLUENT', 'PREBLOOD_INF', 'EXCESS_PT_FLUID', 'PUMP1', 'PUMP2', 'PUMP3', 'PUMP4','ACCESS_PRESSURE', 'FILTER_PRESSURE',
       'EFFLUENT_PRESSURE', 'RETURN_PRESSURE']]

In [ ]:

# Algorithm: Remove Priming and Downtime Segments from CRRT Machine Data

# Goal:
#   Remove non-informative CRRT machine rows corresponding to priming
#   (RUN_TIME = 0) and stagnant downtime periods while preserving
#   valid minute-wise continuity.
#
# Steps:
# 1. Sort data by PID, Visit_no, and Time.
# 2. Within each PID–Visit group, compute time differences (dt).
# 3. Identify removable rows when:
#    - Consecutive rows are exactly 1 minute apart AND
#    - (a) RUN_TIME remains 0 → priming period, OR
#    - (b) EFFLUENT does not change → downtime/stagnation.
# 4. Drop identified rows and return the cleaned dataset.
#
# Output:
#   CRRT machine data with priming and downtime segments removed.


def remove_primingtime(machine_filtered_v1: pd.DataFrame,
                    expected_step: str = "1min") -> pd.DataFrame:
    
    EXPECTED_STEP = pd.to_timedelta(expected_step)

    df = machine_filtered_v1.copy()
    df["Time"] = pd.to_datetime(df["Time"], errors="coerce")
    df = df.sort_values(["PID", "Visit_no", "Time"])

    def _process_group(g: pd.DataFrame) -> pd.DataFrame:
        g = g.sort_values("Time").copy()

        g["dt"] = g["Time"].diff()
        valid_edge = g["dt"].eq(EXPECTED_STEP)
        priming_edge = valid_edge & g["RUN_TIME"].eq(0.0) & g["RUN_TIME"].shift().eq(0.0)
        downtime_edge = valid_edge & g["EFFLUENT"].eq(g["EFFLUENT"].shift())
        drop_mask = priming_edge | downtime_edge

        out = g.loc[~drop_mask].drop(columns=["dt"])
        return out

    filtered = (df.groupby(["PID", "Visit_no"], group_keys=False)
                  .apply(_process_group)
                  .reset_index(drop=True))

    return filtered

filtered_no_downtime = remove_primingtime(machine_filtered_v1, expected_step="1min")


## Machine Data filtration based on 14 days

In [ ]:
## Compute Day Number for each visit
filtered_no_downtime['Date'] = filtered_no_downtime['Time'].dt.date
filtered_no_downtime['Day'] = filtered_no_downtime.groupby(['PID', 'Visit_no'])['Date'].rank(method='dense').astype(int)
filtered_no_downtime.drop(columns='Date',inplace=True)

df_filtered=filtered_no_downtime[['PID', 'Visit_no', 'Day','Time', 'RUN_TIME', 'POST_INFUSION', 'PRE_INFUSION',
       'DIALYSATE', 'EFFLUENT', 'PREBLOOD_INF', 'EXCESS_PT_FLUID', 'PUMP1', 'PUMP2', 'PUMP3', 'PUMP4',
       'ACCESS_PRESSURE', 'FILTER_PRESSURE', 'EFFLUENT_PRESSURE',
       'RETURN_PRESSURE']]

df_filtered = df_filtered[~(df_filtered['Day']>14)]

## Extreme Peak detection and smoothing algorithm

In [ ]:
# Algorithm: Window-based Outlier Detection and Local Smoothing (NaN-aware)
# Goal:
#   Detect and smooth extreme values in CRRT machine signals while optionally
#   handling missing values, preserving local temporal structure per patient visit.
#
# Steps:
# 1. Sort data by PID, Visit_no, and Time; partition time into fixed windows (e.g., 40 min).
# 2. For each (PID, Visit_no) and feature, compute window-wise mean and std.
# 3. Flag outliers using z-score threshold (|z| > threshold), ignoring NaNs.
# 4. Replace outliers with the mean of nearest valid neighbors in time;
#    fallback to (PID, Visit_no)-level mean if neighbors are unavailable.
# 5. Optionally fill NaN values using the same neighbor-based strategy.
# 6. Log all modifications (outliers and NaN fills) for auditability.
#
# Output:
#   - Smoothed CRRT time-series DataFrame
#   - Detailed log of all corrected values


def detect_and_smooth_outliers_skip_nan(
    df, time_col, pid_col, feat_cols,
    visit_col='Visit_no',
    window='40T', threshold=2, min_periods=1, neighbor_count=4,
    fill_nan=True ):

    df2 = df.copy()
    df2[time_col] = pd.to_datetime(df2[time_col], errors='coerce')

    # Sort by PID + Visit_no + Time
    df2 = df2.sort_values([pid_col, visit_col, time_col]).reset_index(drop=False)
    df2 = df2.rename(columns={'index': 'orig_index'})

    df2['timestamp'] = df2[time_col]
    df2['win_id'] = df2['timestamp'].dt.floor(window)

    smoothed = df2.copy()
    logs = []

    print(f"\n{'='*80}")
    print(f"OUTLIER DETECTION & SMOOTHING (with NaN handling)")
    print(f"{'='*80}")
    print(f"Features: {len(feat_cols)}")
    print(f"Window: {window}, Threshold: {threshold}")
    print(f"Neighbor count: {neighbor_count}")
    print(f"Fill NaN: {fill_nan}")
    print(f"{'='*80}\n")

    total_outliers = 0
    total_nans_filled = 0
    total_rows_processed = 0
    total_rows = len(df2)
    outliers_per_feat = {f: 0 for f in feat_cols}

    group_keys = [pid_col, visit_col]

    pid_count = 0
    prev_pid = None
    reported_milestones = set()

    for grp_idx, ((pid, visit), grp) in enumerate(df2.groupby(group_keys, sort=False), 1):
        if grp.empty:
            continue

        if pid != prev_pid:
            pid_count += 1
            prev_pid = pid

        pos_idx = grp.index.to_numpy()
        ts_arr = grp['timestamp'].to_numpy()
        orig_idx_arr = grp['orig_index'].to_numpy()
        idx_arr_local = np.arange(len(grp))

        win_groups = grp.groupby('win_id', sort=False)

        outlier_masks = {}
        nan_masks = {}
        z_values_map = {}

        # PHASE 1: DETECT OUTLIERS (per feature)
        for f in feat_cols:
            s = grp[f]
            nan_mask = s.isna()
            nan_masks[f] = nan_mask.to_numpy()
            stats = win_groups[f].agg(['count', 'mean', 'std']).rename(columns={'count': 'cnt'})
            if min_periods > 1:
                stats.loc[stats['cnt'] < min_periods, ['mean', 'std']] = np.nan

            mean_win = grp['win_id'].map(stats['mean'])
            std_win = grp['win_id'].map(stats['std'])
            z = (s - mean_win) / std_win
            mask = (std_win > 0) & z.abs().gt(threshold) & (~nan_mask)

            outlier_masks[f] = mask.to_numpy()
            z_values_map[f] = z.to_numpy()

        # Fallback mean is visit-level (not PID-level)
        group_means = grp[feat_cols].mean(numeric_only=True)

        half = neighbor_count // 2
        next_count = neighbor_count - half

        # PHASE 2: SMOOTH OUTLIERS & FILL NaN
        for f in feat_cols:
            s_vals = grp[f].to_numpy(dtype=float)
            outliers = outlier_masks[f]
            nan_mask = nan_masks[f]
            valid_mask = (~outliers) & (~nan_mask)
            valid_idx_local = idx_arr_local[valid_mask]
            valid_vals = s_vals[valid_mask]

            if outliers.any():
                out_local = idx_arr_local[outliers]

                for i_local in out_local:
                    insert_pos = np.searchsorted(valid_idx_local, i_local)

                    prev_end = insert_pos
                    prev_start = max(0, prev_end - half)
                    prev_neighbors = valid_vals[prev_start:prev_end]
                    next_start = insert_pos
                    next_end = min(len(valid_idx_local), next_start + next_count)
                    next_neighbors = valid_vals[next_start:next_end]
                    neighbors = np.concatenate([prev_neighbors, next_neighbors])
                    search_radius = neighbor_count
                    max_search = len(valid_idx_local)

                    while neighbors.size < neighbor_count and search_radius < max_search:
                        search_radius += neighbor_count
                        prev_start = max(0, insert_pos - search_radius // 2)
                        next_end = min(len(valid_idx_local), insert_pos + search_radius // 2)
                        prev_neighbors = valid_vals[prev_start:insert_pos]
                        next_neighbors = valid_vals[insert_pos:next_end]
                        neighbors = np.concatenate([prev_neighbors, next_neighbors])

                    if neighbors.size > 0:
                        new_val = float(np.mean(neighbors))
                    else:
                        gm = group_means.get(f, np.nan)
                        new_val = float(gm) if not np.isnan(gm) else s_vals[i_local]

                    smoothed.loc[pos_idx[i_local], f] = new_val
                    total_outliers += 1
                    outliers_per_feat[f] += 1

                    logs.append({
                        'PID': pid,
                        'Visit_no': visit,
                        'index': int(orig_idx_arr[i_local]),
                        'feature': f,
                        'timestamp': ts_arr[i_local],
                        'original_value': float(s_vals[i_local]),
                        'smoothed_value': float(new_val),
                        'zscore': float(z_values_map[f][i_local]) if np.isfinite(z_values_map[f][i_local]) else np.nan,
                        'neighbors_used': int(neighbors.size),
                        'reason': 'outlier'
                    })

            if fill_nan and nan_mask.any():
                nan_local = idx_arr_local[nan_mask]

                for i_local in nan_local:
                    insert_pos = np.searchsorted(valid_idx_local, i_local)

                    prev_end = insert_pos
                    prev_start = max(0, prev_end - half)
                    prev_neighbors = valid_vals[prev_start:prev_end]
                    next_start = insert_pos
                    next_end = min(len(valid_idx_local), next_start + next_count)
                    next_neighbors = valid_vals[next_start:next_end]
                    neighbors = np.concatenate([prev_neighbors, next_neighbors])
                    search_radius = neighbor_count
                    max_search = len(valid_idx_local)

                    while neighbors.size < neighbor_count and search_radius < max_search:
                        search_radius += neighbor_count
                        prev_start = max(0, insert_pos - search_radius // 2)
                        next_end = min(len(valid_idx_local), insert_pos + search_radius // 2)
                        prev_neighbors = valid_vals[prev_start:insert_pos]
                        next_neighbors = valid_vals[insert_pos:next_end]
                        neighbors = np.concatenate([prev_neighbors, next_neighbors])

                    if neighbors.size > 0:
                        new_val = float(np.mean(neighbors))
                    else:
                        gm = group_means.get(f, np.nan)
                        new_val = float(gm) if not np.isnan(gm) else np.nan

                    if not np.isnan(new_val):
                        smoothed.loc[pos_idx[i_local], f] = new_val
                        total_nans_filled += 1

                        logs.append({
                            'PID': pid,
                            'Visit_no': visit,
                            'index': int(orig_idx_arr[i_local]),
                            'feature': f,
                            'timestamp': ts_arr[i_local],
                            'original_value': np.nan,
                            'smoothed_value': float(new_val),
                            'zscore': np.nan,
                            'neighbors_used': int(neighbors.size),
                            'reason': 'missing'
                        })

        total_rows_processed += len(grp)

        # Print every 100 unique PIDs (once per milestone)
        milestone = (pid_count // 100) * 100
        if milestone > 0 and milestone not in reported_milestones:
            reported_milestones.add(milestone)
            print(f"\n--- PIDs processed: {milestone} ---")
            rows_so_far = total_rows_processed
            for f in feat_cols:
                valid_so_far = df2.loc[df2.index[:rows_so_far], f].notna().sum() if rows_so_far > 0 else 1
                pct = 100.0 * outliers_per_feat[f] / rows_so_far if rows_so_far > 0 else 0.0
                print(f"  {f}: {outliers_per_feat[f]:,} smoothed ({pct:.2f}% of rows so far)")

    smoothed_df = smoothed.drop(columns=['win_id', 'timestamp', 'orig_index']).reset_index(drop=True)

    records = pd.DataFrame(logs, columns=[
        'PID', 'Visit_no', 'index', 'feature', 'timestamp', 'original_value',
        'smoothed_value', 'zscore', 'neighbors_used', 'reason'
    ])

    print(f"\n{'='*80}")
    print(f"SMOOTHING COMPLETE")
    print(f"{'='*80}")
    print(f"Total PIDs: {pid_count} | Total rows: {total_rows:,}")
    print(f"Per-feature smoothed outliers:")
    for f in feat_cols:
        pct = 100.0 * outliers_per_feat[f] / total_rows if total_rows > 0 else 0.0
        print(f"  {f}: {outliers_per_feat[f]:,} ({pct:.2f}%)")
    print(f"Total outliers smoothed: {total_outliers:,} ({100.0 * total_outliers / total_rows:.2f}% of all data)")
    print(f"Total NaNs filled: {total_nans_filled:,}")
    print(f"Total changes: {len(records):,}")
    print(f"{'='*80}\n")

    return smoothed_df, records

In [ ]:
feat_cols = ['ACCESS_PRESSURE', 'FILTER_PRESSURE', 'EFFLUENT_PRESSURE', 'RETURN_PRESSURE']
    
print(f"Initial data:")
print(f"  Rows: {len(df_filtered):,}")
print(f"  Missing values per feature:")
for col in feat_cols:
    missing = df_filtered[col].isna().sum()
    print(f"    {col}: {missing:,} ({missing/len(df_filtered)*100:.1f}%)")

smoothed_df, records = detect_and_smooth_outliers_skip_nan(
    df_filtered,
    time_col='Time',
    pid_col='PID',
    feat_cols=feat_cols,
    visit_col='Visit_no',
    window='40T',
    threshold=2,
    min_periods=1,
    neighbor_count=4,
    fill_nan=False  
)


In [ ]:
feat_cols = ['ACCESS_PRESSURE','FILTER_PRESSURE','EFFLUENT_PRESSURE','RETURN_PRESSURE']
df_pressure_cleaned=smoothed_df[feat_cols]

def outlier_smoothing_summary(records, df_source, features):
    rows = []
    for f in features:
        n_smoothed = (records['feature'] == f).sum()              # outliers detected & smoothed
        n_total    = df_source[f].notna().sum()                    # total valid values
        pct        = 100.0 * n_smoothed / n_total if n_total else float('nan')
        rows.append({
            'feature': f,
            'smoothed_outliers': int(n_smoothed),
            'total_valid_values': int(n_total),
            '%_smoothed': round(pct, 3)
        })
    return pd.DataFrame(rows).sort_values('%_smoothed', ascending=False)

summary = outlier_smoothing_summary(records, df_pressure_cleaned, feat_cols)
print(summary.to_string(index=False))

## Separate the features 

In [ ]:
smoothed_df.columns

In [ ]:
feat_fluid = ['PID', 'Visit_no', 'Day', 'Time', 'RUN_TIME', 'POST_INFUSION',
       'PRE_INFUSION', 'DIALYSATE', 'EFFLUENT', 'PREBLOOD_INF',
       'EXCESS_PT_FLUID']

feat_pressure = ['PID', 'Visit_no', 'Day', 'Time','ACCESS_PRESSURE','FILTER_PRESSURE','EFFLUENT_PRESSURE','RETURN_PRESSURE']


fluid_df = smoothed_df[feat_fluid]
pressure_df_wo_peak = smoothed_df[feat_pressure]
pressure_df_w_peak = df_filtered[feat_pressure]



In [ ]:
fluid_df.to_csv(os.path.join(parent_dir, "Processed","Machine","fluid", "CRRT_fluid.csv"), index=False)
pressure_df_wo_peak.to_csv(os.path.join(parent_dir, "Processed","Machine","pressure","denoised", "CRRT_pressure_wo_peak.csv"), index=False)
pressure_df_w_peak.to_csv(os.path.join(parent_dir, "Processed","Machine","pressure","noised", "CRRT_pressure_with_peak.csv"), index=False)